<a href="https://colab.research.google.com/github/Nassemm/NASSEM-MAHARMEH-PORTFOLLIO/blob/main/replication_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3


import numpy as np
import pandas as pd
import statsmodels.api as sm


def ols(y, X):
    Xc = sm.add_constant(X, has_constant="add")
    return sm.OLS(y, Xc, missing="drop").fit()


def main(xlsx_path: str):
    df = pd.read_excel(xlsx_path, sheet_name="MRW1992data").copy()

    # Core variables
    df["ln_y85"] = np.log(df["rgdpw85"])
    df["ln_s"] = np.log(df["i_y"] / 100.0)

    g_plus_delta = 0.05  # MRW g + δ = 0.05
    df["ln_ngd"] = np.log(df["popgrowth"] / 100.0 + g_plus_delta)

    # Human capital
    # Replace 0 with nan for log transformation
    df["ln_school_frac"] = np.log((df["school"] / 100.0).replace(0, np.nan))
    df["ln_school_pct"] = np.log(df["school"].replace(0, np.nan))

    # Samples
    samples = {
        "Non-oil": df.query("n == 1").copy(),
        "Intermediate": df.query("i == 1").copy(),
        "OECD": df.query("o == 1").copy(),
    }

    # Function to generate table rows and display summaries
    def generate_table_rows_and_summaries(samples_dict, table_num):
        rows = []
        for name, d in samples_dict.items():
            print(f"\n--- {name} Sample - Table {table_num} ---")

            if table_num == "I":
                m_unres = ols(d["ln_y85"], d[["ln_s", "ln_ngd"]])
                print(f"\nUnrestricted OLS Summary for {name} (Table I):")
                display(m_unres.summary())

                d["diff"] = d["ln_s"] - d["ln_ngd"]
                m_res = ols(d["ln_y85"], d[["diff"]])
                print(f"\nRestricted OLS Summary for {name} (Table I):")
                display(m_res.summary())


                rows.append(
                    {
                        "Table": table_num,
                        "Sample": name,
                        "Obs": int(m_unres.nobs),
                        "CONSTANT (unres)": m_unres.params.get("const", np.nan),
                        "ln(I/GDP)": m_unres.params.get("ln_s", np.nan),
                        "ln(n+g+δ)": m_unres.params.get("ln_ngd", np.nan),
                        "P-value (ln(I/GDP)) (unres)": m_unres.pvalues.get("ln_s", np.nan),
                        "P-value (ln(n+g+δ)) (unres)": m_unres.pvalues.get("ln_ngd", np.nan),
                        "R2 (unres)": m_unres.rsquared_adj,
                        "SEE (unres)": np.sqrt(m_unres.mse_resid),
                        "CONSTANT (res)": m_res.params.get("const", np.nan),
                        "[ln(I/GDP) - ln(n+g+δ)]": m_res.params.get("diff", np.nan),
                        "P-value ([ln(I/GDP) - ln(n+g+δ)]) (res)": m_res.pvalues.get("diff", np.nan),
                        "R2 (res)": m_res.rsquared_adj,
                        "SEE (res)": np.sqrt(m_res.mse_resid),
                    }
                )
            elif table_num == "II":
                # Use fraction scaling consistently for Table II
                choice_col = "ln_school_frac"

                m_unres = ols(d["ln_y85"], d[["ln_s", "ln_ngd", choice_col]])
                print(f"\nUnrestricted OLS Summary for {name} (Table II, {choice_col}):")
                display(m_unres.summary())

                d["diff1"] = d["ln_s"] - d["ln_ngd"]
                d["diff2"] = d[choice_col] - d["ln_ngd"]
                m_res = ols(d["ln_y85"], d[["diff1", "diff2"]])
                print(f"\nRestricted OLS Summary for {name} (Table II, {choice_col}):")
                display(m_res.summary())

                rows.append(
                    {
                        "Table": table_num,
                        "Sample": name,
                        "Obs": int(m_unres.nobs),
                        "CONSTANT (unres)": m_unres.params.get("const", np.nan),
                        "ln(I/GDP)": m_unres.params.get("ln_s", np.nan),
                        "ln(n+g+δ)": m_unres.params.get("ln_ngd", np.nan),
                        f"ln(SCHOOL) ({'pct' if choice_col.endswith('pct') else 'frac'})": m_unres.params.get(
                            choice_col, np.nan
                        ),
                        "P-value (ln(I/GDP)) (unres)": m_unres.pvalues.get("ln_s", np.nan),
                        "P-value (ln(n+g+δ)) (unres)": m_unres.pvalues.get("ln_ngd", np.nan),
                        f"P-value (ln(SCHOOL) ({'pct' if choice_col.endswith('pct') else 'frac'})) (unres)": m_unres.pvalues.get(choice_col, np.nan),
                        "R2 (unres)": m_unres.rsquared_adj,
                        "SEE (unres)": np.sqrt(m_unres.mse_resid),
                        "CONSTANT (res)": m_res.params.get("const", np.nan),
                        "[ln(I/GDP) - ln(n+g+δ)]": m_res.params.get("diff1", np.nan),
                        "[ln(SCHOOL) - ln(n+g+δ)]": m_res.params.get("diff2", np.nan),
                        "P-value ([ln(I/GDP) - ln(n+g+δ)]) (res)": m_res.pvalues.get("diff1", np.nan),
                        "P-value ([ln(SCHOOL) - ln(n+g+δ)]) (res)": m_res.pvalues.get("diff2", np.nan),
                        "R2 (res)": m_res.rsquared_adj,
                        "SEE (res)": np.sqrt(m_res.mse_resid),
                    }
                )
        return pd.DataFrame(rows)

    # Table I: Textbook Solow
    table_I = generate_table_rows_and_summaries(samples, "I")

    # Table II: Augmented Solow
    table_II = generate_table_rows_and_summaries(samples, "II")

    #SCHOOL scaling (fraction)
    choice_col = "ln_school_frac"


    # Display Lines
    print("\n MRW Table I Replication:")
    display(table_I.T)
    print("\n MRW Table II Replication:")
    display(table_II.T)
    print(f"\nChosen SCHOOL scaling for Table II: {'percent (0–100)' if choice_col.endswith('pct') else 'fraction (0–1)'}")

    return table_I, table_II
main("MRW1992data.xlsx")


--- Non-oil Sample - Table I ---

Unrestricted OLS Summary for Non-oil (Table I):


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 ln_y85   R-squared:                       0.601
Model:                            OLS   Adj. R-squared:                  0.592
Method:                 Least Squares   F-statistic:                     71.51
Date:                Tue, 23 Sep 2025   Prob (F-statistic):           1.13e-19
Time:                        18:52:48   Log-Likelihood:                -101.04
No. Observations:                  98   AIC:                             208.1
Df Residuals:                      95   BIC:                             215.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.4299      1.584      3.428      0.001       2.285       8.574
ln_s           1.4240      0.143      9.951      0.000       1.140       1.708
ln_ngd        -1.9898      0.563     -3.532      0.001      -3.108      -0.871
==============================================================================
Omnibus:                        2.881   Durbin-Watson:                   1.453
Prob(Omnibus):                  0.237   Jarque-Bera (JB):                2.631
Skew:                          -0.401   Prob(JB):                        0.268
Kurtosis:                       2.976   Cond. No.                         81.7
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""


Restricted OLS Summary for Non-oil (Table I):


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 ln_y85   R-squared:                       0.597
Model:                            OLS   Adj. R-squared:                  0.593
Method:                 Least Squares   F-statistic:                     142.4
Date:                Tue, 23 Sep 2025   Prob (F-statistic):           1.13e-20
Time:                        18:52:48   Log-Likelihood:                -101.46
No. Observations:                  98   AIC:                             206.9
Df Residuals:                      96   BIC:                             212.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          6.8724      0.121     56.995      0.000       6.633       7.112
diff           1.4880      0.125     11.934      0.000       1.241       1.735
==============================================================================
Omnibus:                        4.010   Durbin-Watson:                   1.421
Prob(Omnibus):                  0.135   Jarque-Bera (JB):                3.720
Skew:                          -0.477   Prob(JB):                        0.156
Kurtosis:                       3.015   Cond. No.                         3.15
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""


--- Intermediate Sample - Table I ---

Unrestricted OLS Summary for Intermediate (Table I):


...

,0,1,2
Table,I,I,I
Sample,Non-oil,Intermediate,OECD
Obs,98,75,22
CONSTANT (unres),5.429883,5.345865,8.020607
ln(I/GDP),1.424014,1.317553,0.49989
ln(n+g+δ),-1.989774,-2.017199,-0.741921
P-value (ln(I/GDP)) (unres),0.0,0.0,0.263572
P-value (ln(n+g+δ)) (unres),0.000639,0.000322,0.394839
R2 (unres),0.592462,0.587767,0.011813
SEE (unres),0.68907,0.610641,0.377396



 MRW Table II Replication:


,0,1,2
Table,II,II,II
Sample,Non-oil,Intermediate,OECD
Obs,98,75,22
CONSTANT (unres),6.844414,7.791314,8.636893
ln(I/GDP),0.696709,0.700367,0.276134
ln(n+g+δ),-1.745247,-1.49978,-1.075506
ln(SCHOOL) (frac),0.654459,0.730549,0.767571
P-value (ln(I/GDP)) (unres),0.000001,0.000015,0.486805
P-value (ln(n+g+δ)) (unres),0.000062,0.000396,0.171947
P-value (ln(SCHOOL) (frac)) (unres),0.0,0.0,0.017462



Chosen SCHOOL scaling for Table II: fraction (0–1)


(  Table        Sample  Obs  CONSTANT (unres)  ln(I/GDP)  ln(n+g+δ)  \
 0     I       Non-oil   98          5.429883   1.424014  -1.989774   
 1     I  Intermediate   75          5.345865   1.317553  -2.017199   
 2     I          OECD   22          8.020607   0.499890  -0.741921   
 
    P-value (ln(I/GDP)) (unres)  P-value (ln(n+g+δ)) (unres)  R2 (unres)  \
 0                 2.104645e-16                     0.000639    0.592462   
 1                 5.383214e-11                     0.000322    0.587767   
 2                 2.635722e-01                     0.394839    0.011813   
 
    SEE (unres)  CONSTANT (res)  [ln(I/GDP) - ln(n+g+δ)]  \
 0     0.689070        6.872370                 1.487994   
 1     0.610641        7.092921                 1.430955   
 2     0.377396        8.624380                 0.553845   
 
    P-value ([ln(I/GDP) - ln(n+g+δ)]) (res)  R2 (res)  SEE (res)  
 0                             1.134081e-20  0.593166   0.688475  
 1                             7